# Vision Encoder Test Notebook

This notebook tests the `MobileNetV2FeatureExtractor` class and its integration with the FlappyBird evaluator.

## Test Coverage:
1. Basic initialization with different feature sizes
2. Feature extraction with synthetic images
3. Feature extraction with actual FlappyBird frames
4. Output shape and data type verification
5. Consistency checks (same frame → same features)
6. Integration with FlappyBirdEvaluator


In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from vision_encoder import MobileNetV2FeatureExtractor

# Check if dependencies are available
try:
    import torch
    import torchvision
    print(f"✓ PyTorch version: {torch.__version__}")
    print(f"✓ Torchvision version: {torchvision.__version__}")
    print(f"✓ CUDA available: {torch.cuda.is_available()}")
except ImportError as e:
    print(f"✗ PyTorch not available: {e}")

try:
    import cv2
    print(f"✓ OpenCV version: {cv2.__version__}")
except ImportError as e:
    print(f"✗ OpenCV not available: {e}")

try:
    import gymnasium as gym
    import flappy_bird_env
    print(f"✓ Gymnasium and FlappyBird environment available")
except ImportError as e:
    print(f"✗ Gymnasium/FlappyBird not available: {e}")


## Test 1: Basic Initialization

Test initialization with different feature sizes.


In [ ]:
# Test initialization with different feature sizes
feature_sizes = [128, 256, 512, 1280]

extractors = {}
for size in feature_sizes:
    try:
        extractor = MobileNetV2FeatureExtractor(feature_size=size)
        extractors[size] = extractor
        print(f"✓ Successfully initialized extractor with feature_size={size}")
        print(f"  Device: {extractor.device}")
        print(f"  Has projection: {extractor.projection is not None}")
    except Exception as e:
        print(f"✗ Failed to initialize extractor with feature_size={size}: {e}")

print(f"\n✓ Initialized {len(extractors)} extractors successfully")


## Test 2: Feature Extraction with Synthetic Images

Test feature extraction with synthetic RGB images of different sizes.


In [ ]:
# Create synthetic test images
test_images = {
    "800x576 (FlappyBird size)": np.random.randint(0, 256, (800, 576, 3), dtype=np.uint8),
    "224x224 (MobileNetV2 input)": np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8),
    "100x100 (small)": np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8),
    "1920x1080 (large)": np.random.randint(0, 256, (1080, 1920, 3), dtype=np.uint8),
}

# Use the 256D extractor for testing
extractor_256 = extractors.get(256)
if extractor_256 is None:
    extractor_256 = MobileNetV2FeatureExtractor(feature_size=256)

print("Testing feature extraction with synthetic images:\n")
for name, image in test_images.items():
    try:
        features = extractor_256.extract_features(image)
        print(f"✓ {name}:")
        print(f"  Input shape: {image.shape}")
        print(f"  Output shape: {features.shape}")
        print(f"  Output dtype: {features.dtype}")
        print(f"  Feature range: [{features.min():.4f}, {features.max():.4f}]")
        print(f"  Feature mean: {features.mean():.4f}")
        print(f"  Feature std: {features.std():.4f}")
        print()
    except Exception as e:
        print(f"✗ {name}: Failed - {e}\n")


## Test 3: Consistency Check

Verify that the same frame produces the same features (deterministic behavior).


In [ ]:
# Test consistency: same frame should produce same features
test_image = np.random.randint(0, 256, (800, 576, 3), dtype=np.uint8)

# Extract features multiple times
n_runs = 5
features_list = []
for i in range(n_runs):
    features = extractor_256.extract_features(test_image)
    features_list.append(features)

# Check if all features are identical
all_same = all(np.allclose(features_list[0], f, rtol=1e-5) for f in features_list[1:])

if all_same:
    print(f"✓ Consistency check passed: {n_runs} runs produced identical features")
else:
    print(f"✗ Consistency check failed: features differ between runs")
    # Show differences
    for i in range(1, n_runs):
        diff = np.abs(features_list[0] - features_list[i])
        print(f"  Max difference (run 0 vs run {i}): {diff.max():.6f}")
        print(f"  Mean difference: {diff.mean():.6f}")


## Test 4: Different Feature Sizes

Test that different feature sizes produce correctly sized outputs.


In [ ]:
# Test all feature sizes with the same image
test_image = np.random.randint(0, 256, (800, 576, 3), dtype=np.uint8)

print("Testing different feature sizes:\n")
for size in feature_sizes:
    if size in extractors:
        extractor = extractors[size]
        try:
            features = extractor.extract_features(test_image)
            assert features.shape == (size,), f"Expected shape ({size},), got {features.shape}"
            print(f"✓ feature_size={size}: Output shape {features.shape} ✓")
        except Exception as e:
            print(f"✗ feature_size={size}: Failed - {e}")


## Test 5: FlappyBird Environment Integration

Test feature extraction with actual FlappyBird game frames.


In [ ]:
# Test with actual FlappyBird environment
try:
    env = gym.make("FlappyBird-v0", render_mode="rgb_array")
    observation, _ = env.reset()
    observation = np.asarray(observation, dtype=np.uint8)
    
    print(f"✓ FlappyBird environment initialized")
    print(f"  Observation shape: {observation.shape}")
    print(f"  Observation dtype: {observation.dtype}")
    print(f"  Observation range: [{observation.min()}, {observation.max()}]")
    
    # Extract features from the frame
    features = extractor_256.extract_features(observation)
    print(f"\n✓ Feature extraction successful:")
    print(f"  Feature shape: {features.shape}")
    print(f"  Feature dtype: {features.dtype}")
    print(f"  Feature range: [{features.min():.4f}, {features.max():.4f}]")
    print(f"  Feature mean: {features.mean():.4f}")
    print(f"  Feature std: {features.std():.4f}")
    
    # Test with multiple frames
    print(f"\nTesting with multiple frames:")
    features_list = []
    for i in range(5):
        observation, _, terminated, truncated, _ = env.step(0)  # NOOP action
        observation = np.asarray(observation, dtype=np.uint8)
        features = extractor_256.extract_features(observation)
        features_list.append(features)
        if terminated or truncated:
            observation, _ = env.reset()
    
    print(f"  ✓ Extracted features from {len(features_list)} frames")
    print(f"  All features have shape {features_list[0].shape}")
    
    # Check feature variation across frames
    features_array = np.array(features_list)
    print(f"  Feature variation across frames:")
    print(f"    Mean std across features: {features_array.std(axis=0).mean():.4f}")
    print(f"    Max std across features: {features_array.std(axis=0).max():.4f}")
    
    env.close()
    
except Exception as e:
    print(f"✗ FlappyBird environment test failed: {e}")
    import traceback
    traceback.print_exc()


## Test 6: FlappyBirdEvaluator Integration

Test the feature_vector strategy in FlappyBirdEvaluator.


In [ ]:
# Test FlappyBirdEvaluator with feature_vector strategy
from evaluator import FlappyBirdEvaluator
from memory_system import MemoryConfig, MemoryBank
from individual import Individual
from instruction_set import InstructionSet
from operation import SCALAR_OPS, VECTOR_OPS

try:
    # Create evaluator with feature_vector strategy
    evaluator = FlappyBirdEvaluator(
        env_id="FlappyBird-v0",
        episodes=1,
        max_steps=10,  # Short test
        output_register=7,
        render_mode="rgb_array",  # Headless mode - returns observations without opening window
        patch_strategy="feature_vector",
        feature_vector_size=256
    )
    
    print(f"✓ FlappyBirdEvaluator created with feature_vector strategy")
    print(f"  Feature vector size: {evaluator.feature_vector_size}")
    print(f"  Feature extractor initialized: {evaluator.feature_extractor is not None}")
    
    # Create a simple individual for testing
    memory_cfg = MemoryConfig(
        n_scalar=8,
        n_vector=4,
        n_matrix=0,
        n_obs_scalar=0,
        n_obs_vector=1,  # One vector observation register
        n_obs_matrix=0,
        vector_size=256,  # Must match feature_vector_size
        matrix_shape=(1, 1),
    )
    
    # Create template memory from config
    template_memory = MemoryBank(
        n_scalar=memory_cfg.n_scalar,
        n_vector=memory_cfg.n_vector,
        n_matrix=memory_cfg.n_matrix,
        n_obs_scalar=memory_cfg.n_obs_scalar,
        n_obs_vector=memory_cfg.n_obs_vector,
        n_obs_matrix=memory_cfg.n_obs_matrix,
        vector_size=memory_cfg.vector_size,
        matrix_shape=memory_cfg.matrix_shape,
    )
    
    instr_set = InstructionSet([op() for op in SCALAR_OPS + VECTOR_OPS], template_memory)
    individual = Individual.random(instr_set, memory_cfg, program_length=5, rng=np.random.default_rng(0))
    
    print(f"\n✓ Individual created with memory config:")
    print(f"  n_obs_vector: {memory_cfg.n_obs_vector}")
    print(f"  vector_size: {memory_cfg.vector_size}")
    
    # Test a single episode
    print(f"\nRunning test episode...")
    fitness = evaluator.evaluate(individual)
    print(f"✓ Test episode completed")
    print(f"  Fitness: {fitness}")
    
    evaluator.close()
    
except Exception as e:
    print(f"✗ FlappyBirdEvaluator integration test failed: {e}")
    import traceback
    traceback.print_exc()


## Test 7: Visualization

Visualize feature extraction process and feature statistics.


In [ ]:
# Visualize feature extraction
try:
    env = gym.make("FlappyBird-v0", render_mode="rgb_array")
    observation, _ = env.reset()
    observation = np.asarray(observation, dtype=np.uint8)
    
    # Extract features
    features = extractor_256.extract_features(observation)
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Original frame
    axes[0, 0].imshow(observation)
    axes[0, 0].set_title(f"Original Frame\nShape: {observation.shape}")
    axes[0, 0].axis('off')
    
    # Feature vector histogram
    axes[0, 1].hist(features, bins=50, edgecolor='black')
    axes[0, 1].set_title(f"Feature Vector Distribution\nShape: {features.shape}")
    axes[0, 1].set_xlabel("Feature Value")
    axes[0, 1].set_ylabel("Frequency")
    axes[0, 1].grid(True, alpha=0.3)
    
    # Feature vector line plot (first 50 features)
    axes[1, 0].plot(features[:50], marker='o', markersize=3)
    axes[1, 0].set_title("First 50 Features")
    axes[1, 0].set_xlabel("Feature Index")
    axes[1, 0].set_ylabel("Feature Value")
    axes[1, 0].grid(True, alpha=0.3)
    
    # Feature statistics
    stats_text = f"""Feature Statistics:
    
Shape: {features.shape}
Dtype: {features.dtype}
Min: {features.min():.4f}
Max: {features.max():.4f}
Mean: {features.mean():.4f}
Std: {features.std():.4f}
Median: {np.median(features):.4f}
"""
    axes[1, 1].text(0.1, 0.5, stats_text, fontsize=12, 
                     verticalalignment='center', family='monospace')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    env.close()
    
except Exception as e:
    print(f"✗ Visualization failed: {e}")
    import traceback
    traceback.print_exc()


## Test 8: Performance Benchmark

Measure feature extraction speed.


In [ ]:
# Performance benchmark
import time

test_image = np.random.randint(0, 256, (800, 576, 3), dtype=np.uint8)
n_iterations = 100

print(f"Benchmarking feature extraction ({n_iterations} iterations)...\n")

# Warmup
for _ in range(5):
    _ = extractor_256.extract_features(test_image)

# Benchmark
start_time = time.time()
for _ in range(n_iterations):
    _ = extractor_256.extract_features(test_image)
end_time = time.time()

total_time = end_time - start_time
avg_time = total_time / n_iterations
fps = 1.0 / avg_time

print(f"Results:")
print(f"  Total time: {total_time:.4f} seconds")
print(f"  Average time per frame: {avg_time*1000:.2f} ms")
print(f"  Throughput: {fps:.2f} frames/second")
print(f"  Device: {extractor_256.device}")

# Compare different feature sizes
print(f"\nComparing different feature sizes:")
for size in [128, 256, 512, 1280]:
    if size in extractors:
        extractor = extractors[size]
        start_time = time.time()
        for _ in range(n_iterations):
            _ = extractor.extract_features(test_image)
        end_time = time.time()
        avg_time = (end_time - start_time) / n_iterations
        print(f"  feature_size={size:4d}: {avg_time*1000:6.2f} ms/frame ({1.0/avg_time:.2f} fps)")


## Summary

All tests completed! The vision encoder is working correctly.
